In [ ]:
# wird nur bei der ersten Ausführung benötigt

pip install openai 

<br>
<br>

# Lernmodul: Interaktion mit Large Language Models (LLMs)
### Kurs: KIT 11 (FOSBOS Bayern)

---
## 0. Einführungsaufgabe: Das "Blackbox"-Experiment
Bevor wir programmieren, überlegen wir uns, wie eine KI eigentlich "denkt".

**Arbeitsauftrag:**
Stell dir vor, du spielst ein Spiel: Ich nenne ein Wort, und du musst spontan ein dazu passendes Wort nennen (Beispiel: "Der -> Ball -> ist..." -> "...rund").

Spielt dieses Spiel innerhalb der Klasse, sodass jeder einmal an der Reihe war, ob ein Satz ergänzt oder neu angefangen wird, soll keine Rolle spielen. Notiert den entstehenden "Satz", sodass jeder den Kontext mitverfolgen kann.
   

Wie lautet das entstandene Satzgefüge?







Recherchiere kurz den Begriff "Stochastische Papageien" (Stochastic Parrots). Warum wird dieser Begriff oft kritisch für Sprachmodelle verwendet? Notiere deine Antwort in der folgenden Zelle.

Antwort:





<br>
<br>
<br>
<br>
<br>
<br>

In [ ]:
# Festlegungen

from openai import OpenAI

OPENAI_API_KEY=""

mein_client = OpenAI(api_key=OPENAI_API_KEY)
max_tokens = 100
kreatititaet = 0 # Werte zwischen 0 und 2 sind möglich; 2 ist maximale Kreativitaet der Antwort

## 1.0 Formuliere die erste Anfrage an das System

In [ ]:
systempromt = "Du bist ein hilfreicher Tutor für Informatik. Antworte immer total genervt und trotzig."

userprompt1 = "Was ist ein Algorithmus?"

messages1 = [
    {"role": "system", "content": systempromt},
    {"role": "user", "content": userprompt1},
]

In [ ]:
antwort1 = mein_client.responses.create(
    model="gpt-4.1-mini",
    input=messages1, max_output_tokens=max_tokens, temperature=kreatititaet)

print(antwort1.output[0].content[0].text)

In [ ]:
print(antwort1)

## 1.1 Aufgaben

* Experimentiere mit der Systemantwort.
  
  Der Chatbot soll...
   * höflich antworten
   * jede Antwort in eine bestimmte Sprache übersetzen
   * wie ein Pirat sprechen
   

* Experimentiere mit dem Wert der **Kreativität** (Temperatur). Beschreibe die Auswirkung der gewählten Niveaustufen von 0 - 2 in Bezug auf die Antwort des Systems.

* Ändere ein paar Mal den Wert von **userprompt1** und lasse jedes Mal eine Antwort generieren. Besitzt der Chatbot deiner Meinung nach ein Gedächtnis?

* Die maximal ausgegebene Anzahl von Tokens einer Antwort kann dem System auch über einen Prompt vorgegeben werden. Beschreibe den Vorteil der hier verwendeten Methode (Verwendung des Parameters max_output_tokens). Finde den minimalen Wert, den das System akzeptiert.

Antworten:







<br>
<br>

## 2.0 Stelle dem System eine weitere Frage mit Bezug zur ersten Antwort

In [ ]:
userprompt2="Sei nicht so garstig!"

In [ ]:
messages2 = [
    {"role": "system", "content": systempromt},
    {"role": "user", "content": userprompt1},
    {"role": "assistant", "content": antwort1.output[0].content[0].text},
    {"role": "user", "content": userprompt2},
]

In [ ]:
antwort2 = mein_client.responses.create(
    model="gpt-4.1-mini",
    input=messages2, max_output_tokens=max_tokens, temperature=kreatititaet)

In [ ]:
print(antwort2.output[0].content[0].text)

In [ ]:
print(antwort2)

<br>
<br>
<br>

## 2.1 Aufgaben
* Notiere in der nächsten Zelle die verschiedenen Rollen, die im obigen Nachrichtenverlauf **messages2** zu sehen sind, sowie deren Bedeutung.
  

Antwort:








* Finde heraus, unter welchen Voraussetzungen sich das Sprachmodell an Details der Konversation "erinnern" kann. Lösche zu diesem Zweck aus **messages2** die **user**- oder **assistant**-Anfragen heraus.

Antwort:











## Hintergrund: Wie verarbeitet ein Sprachmodell einen Prompt?

Ein Modell wie GPT funktioniert nicht wie eine Datenbank-Abfrage, sondern wie ein hochkomplexer Autocomplete-Mechanismus:

* **Tokenisierung:** Dein Text wird in Fragmente (**Tokens**, **https://platform.openai.com/tokenizer**) zerlegt. Ein Token entspricht etwa 4 Zeichen oder einem halben Wort.
* **Statelessness (Zustandslosigkeit):** Das Modell hat kein eigenes Gedächtnis. Es "vergisst" alles, sobald eine Antwort generiert wurde. 
* **Kontextfenster:** Damit der Bot sich an den Chat-Verlauf erinnert, müssen wir ihm bei **jeder** neuen Anfrage den **gesamten bisherigen Verlauf** mitschicken.
* **Wahrscheinlichkeit:** Das Modell berechnet für das nächste Token die höchste Wahrscheinlichkeit basierend auf dem Kontext. Der Parameter `temperature` bestimmt dabei, wie stark auch weniger wahrscheinliche (kreativere) Wörter gewählt werden.

<br>
<br>
<br>
<br>



<br>
<br>

## 3. beliebig lange Frage-Antwort-Sequenz

Das folgende Beispiel zeigt, wie nach den Fragen 1 und 2 ein Chat-Verlauf weiter geführt werden könnte. Dies kennt man i. A. von Webseiten, die Chat-Bots anbieten.


In [ ]:
chatverlauf = []
chatverlauf.append({"role": "system", "content": systempromt})
chatverlauf.append({"role": "user", "content": userprompt1})                                
chatverlauf.append({"role": "assistant", "content": antwort1.output[0].content[0].text})
chatverlauf.append({"role": "user", "content": userprompt2})
chatverlauf.append({"role": "assistant", "content": antwort2.output[0].content[0].text})

neue_usereingabe = ""
antwort_neu = ""   

while neue_usereingabe != "exit":
    neue_usereingabe = input()
    if neue_usereingabe!= "exit":
        neue_Nachricht = chatverlauf.append({"role": "user", "content": neue_usereingabe}) 
        antwort_neu = mein_client.responses.create(
        model="gpt-4.1-mini",
        input=chatverlauf, max_output_tokens=max_tokens, temperature=kreatititaet)
        print(antwort_neu.output[0].content[0].text)
        chatverlauf.append({"role": "assistant", "content": antwort_neu.output[0].content[0].text})             
                 
                 


In [ ]:
print(chatverlauf)

In [ ]:
print(antwort_neu)

<br>
<br>
<br>

## 3.1 Aufgabe
Ändere den Code unter 3. so ab, dass eine völlig neue Konversation (ohne den Bezug zu den ersten beiden Anfragen sowie den zugehörigen Antworten) geführt werden kann. Erhalte aber den Systemprompt.

Antwort:





